Testing Inseption-LSTM model with different dataset

Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, precision_recall_curve
from tensorflow.keras.models import load_model

Load Models & Datasets

In [3]:
# Load the trained model
model = load_model("/Users/pasan_diksura/Documents/SoftwareDevelopment/Projects/Signature-Forgery-Detection-System/SignatureForgeryDetectionSystem/MachineLearning/Model/signature_forgery_detection_model.h5")

In [ ]:
# Paths to testing dataset
advance_genuine_path = "/Users/pasan_diksura/Documents/SoftwareDevelopment/Projects/Signature-Forgery-Detection-System/SignatureForgeryDetectionSystem/MachineLearning/Dataset/Processing/AdvanceTestingDataset/Genuine"
advance_forged_path = "/Users/pasan_diksura/Documents/SoftwareDevelopment/Projects/Signature-Forgery-Detection-System/SignatureForgeryDetectionSystem/MachineLearning/Dataset/Processing/AdvanceTestingDataset/Forged"

Definitions

In [ ]:
def preprocess_image(image_path):
    """Preprocesses an image for model input."""
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (256, 128))  # Resize to match training input
    img = img / 255.0  # Normalize
    img = np.expand_dims(img, axis=-1)  # Add channel dimension
    return img

In [ ]:
def load_testing_data(folder, label):
    """Loads images from a folder and assigns a label."""
    data, labels = [], []
    for file_name in os.listdir(folder):
        image_path = os.path.join(folder, file_name)
        img = preprocess_image(image_path)
        data.append(img)
        labels.append(label)
    return np.array(data), np.array(labels)

Process

In [ ]:
# Load genuine and forged images
genuine_images, genuine_labels = load_testing_data(advance_genuine_path, 1)
forged_images, forged_labels = load_testing_data(advance_forged_path, 0)

# Combine datasets
test_images = np.concatenate((genuine_images, forged_images), axis=0)
test_labels = np.concatenate((genuine_labels, forged_labels), axis=0)

# Make predictions
predictions = model.predict(test_images)
predicted_labels = (predictions > 0.5).astype(int).flatten()


Evaluations

In [ ]:
# Evaluate performance
print("Classification Report:\n", classification_report(test_labels, predicted_labels))


In [ ]:
# Confusion Matrix
cm = confusion_matrix(test_labels, predicted_labels)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Forged', 'Genuine'], yticklabels=['Forged', 'Genuine'])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()


In [ ]:
# ROC Curve
fpr, tpr, _ = roc_curve(test_labels, predictions)
roc_auc = auc(fpr, tpr)
plt.plot(fpr, tpr, color='blue', label=f'ROC Curve (AUC = {roc_auc:.2f}')
plt.plot([0, 1], [0, 1], 'r--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Receiver Operating Characteristic (ROC) Curve")
plt.legend()
plt.show()



In [ ]:
# Precision-Recall Curve
precision, recall, _ = precision_recall_curve(test_labels, predictions)
plt.plot(recall, precision, color='green', label="Precision-Recall Curve")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.legend()
plt.show()


In [ ]:
# Distribution of Predictions
sns.histplot(predictions, bins=20, kde=True)
plt.xlabel("Prediction Confidence")
plt.ylabel("Frequency")
plt.title("Distribution of Model Predictions")
plt.show()

print("Advanced testing completed with visualizations!")